In [5]:
!pip install spacy


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
!pip install --upgrade typing_extensions


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
# Download directly using pip
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 528.8 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import re
from typing import Dict, List, Tuple
from collections import defaultdict
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
class HotelIntentClassifier:
    """
    Rule-based intent classifier for hotel travel assistant using keyword matching.
    """
    
    def __init__(self):
        # Define intent keywords and patterns
        self.intent_keywords = {
            'VISA_INQUIRY': {
                'keywords': ['visa', 'visa requirements', 'need visa', 'visa type', 
                           'visa application', 'entry requirements', 'travel documents'],
                'patterns': [
                    r'\b(do|does|will|can).*(need|require).*(visa|entry)\b',
                    r'\bvisa.*(from|to|between)\b',
                    r'\b(travel|entry).*(requirement|document)\b'
                ]
            },
            
            'HOTEL_RECOMMENDATION': {
                'keywords': ['recommend', 'suggestion', 'suggest', 'best hotel', 
                           'which hotel', 'should i book', 'good hotel', 'top hotel',
                           'where should i stay', 'hotel for me'],
                'patterns': [
                    r'\b(recommend|suggest).*(hotel|place|stay)\b',
                    r'\b(best|top|good).*(hotel|place)\b',
                    r'\b(which|what).*(hotel|place).*(should|recommend|best)\b',
                    r'\bhotel for\b'
                ]
            },
            
            'DEMOGRAPHIC_RECOMMENDATION': {
                'keywords': ['solo female', 'male traveler', 'young woman', 'elderly',
                           'senior', 'young male', 'business traveler', 'family',
                           'couple', 'age', 'years old', 'female solo', 'male solo'],
                'patterns': [
                    r'\b\d+\s*(year|yr)s?\s*old\b',
                    r'\b(male|female|woman|man).*(traveler|travel|solo)\b',
                    r'\b(young|elderly|senior).*(traveler|woman|man|male|female)\b',
                    r'\b(business|leisure|solo|family|couple).*(traveler|travel)\b',
                    r'\bi\'?m\s*a\s*\d+\b'
                ]
            },
            
            'HOTEL_SEARCH': {
                'keywords': ['find', 'show', 'list', 'search', 'hotels in', 
                           'available', 'look for', 'looking for', 'get hotels',
                           'hotels with', 'hotels that have'],
                'patterns': [
                    r'\b(find|show|list|search|get).*(hotel|place)\b',
                    r'\bhotels?\s*(in|at|near|with)\b',
                    r'\b(available|any).*(hotel|place)\b',
                    r'\b\d\s*star\s*hotel\b'
                ]
            },
            
            'REVIEW_QUERY': {
                'keywords': ['reviews', 'feedback', 'what do people say', 'comments',
                           'opinions', 'ratings', 'what travelers think', 'testimonials',
                           'customer feedback'],
                'patterns': [
                    r'\b(review|rating|feedback|comment|opinion)s?\b',
                    r'\bwhat (do|did).*(say|think|rate)\b',
                    r'\b(people|traveler|customer|guest)s?.*(say|think|rate|feedback)\b',
                    r'\b(positive|negative).*(review|feedback)\b'
                ]
            },
            
            'COMPARISON': {
                'keywords': ['compare', 'comparison', 'versus', 'vs', 'difference between',
                           'better than', 'which is better', 'contrast'],
                'patterns': [
                    r'\bcompare.*(hotel|rating|score)\b',
                    r'\b(vs|versus|or)\b',
                    r'\bdifference between\b',
                    r'\b(better|worse).*(than|compare)\b',
                    r'\bwhich (is|has|have).*(better|higher|best)\b'
                ]
            },
            
            'STATISTICAL_QUERY': {
                'keywords': ['average', 'total', 'how many', 'count', 'highest',
                           'lowest', 'most', 'least', 'percentage', 'statistics',
                           'mean', 'median'],
                'patterns': [
                    r'\b(average|mean|median|total|sum)\b',
                    r'\bhow many\b',
                    r'\b(highest|lowest|most|least|top|bottom)\b',
                    r'\bpercentage of\b',
                    r'\bcount (of|the)\b'
                ]
            },
            
            'TRAVELER_ANALYSIS': {
                'keywords': ['traveler type', 'demographics', 'age group', 
                           'gender distribution', 'who stays', 'what type of traveler',
                           'traveler pattern', 'visitor demographics'],
                'patterns': [
                    r'\b(traveler|visitor|guest)s?.*(type|demographic|pattern)\b',
                    r'\b(age group|gender|type).*(stay|visit|travel)\b',
                    r'\bwho (stay|visit|travel)s?\b',
                    r'\b(distribution|percentage).*(traveler|guest|age|gender)\b'
                ]
            },
            
            'LOCATION_QUERY': {
                'keywords': ['cities', 'countries', 'locations', 'where', 'places',
                           'destinations', 'which city', 'which country'],
                'patterns': [
                    r'\b(which|what).*(cit|countr|location|place|destination)\b',
                    r'\bhow many.*(cit|countr|location)\b',
                    r'\b(popular|top).*(cit|countr|destination)\b',
                    r'\bwhere (do|did|can)\b'
                ]
            },
        }
        
        # Priority order for intents (higher priority checked first)
        self.intent_priority = [
            'VISA_INQUIRY',
            'DEMOGRAPHIC_RECOMMENDATION',
            'HOTEL_RECOMMENDATION',
            'COMPARISON',
            'REVIEW_QUERY',
            'TRAVELER_ANALYSIS',
            'STATISTICAL_QUERY',
            'HOTEL_SEARCH',
            'LOCATION_QUERY',
        ]
    
    def preprocess_text(self, text: str) -> str:
        """
        Preprocess user input text for classification.
        
        Args:
            text: Raw user input
            
        Returns:
            Preprocessed text (lowercase, normalized)
        """
        # Convert to lowercase
        text = text.lower().strip()
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        
        # Remove special characters but keep important punctuation
        text = re.sub(r'[^\w\s\-?!,.]', '', text)
        
        return text
    
    #replace this method with name entitiy recognition from nltk or el lib el tanya
    def extract_entities_hints(self, text: str) -> Dict[str, List[str]]:
        """
        Extract entity hints that can help with intent classification.
        
        Args:
            text: Preprocessed text
            
        Returns:
            Dictionary of entity types and their values
        """
        entities = defaultdict(list)
        
        # Age patterns
        age_pattern = r'\b(\d+)\s*(year|yr)s?\s*old\b'
        age_matches = re.findall(age_pattern, text)
        if age_matches:
            entities['age'] = [match[0] for match in age_matches]
        
        # Gender patterns
        if re.search(r'\b(male|man|men|boy)\b', text):
            entities['gender'].append('male')
        if re.search(r'\b(female|woman|women|girl|lady)\b', text):
            entities['gender'].append('female')
        
        # Traveler type patterns
        traveler_types = ['business', 'solo', 'family', 'couple']
        for t_type in traveler_types:
            if re.search(rf'\b{t_type}\b', text):
                entities['traveler_type'].append(t_type)
        
        # Star rating patterns
        star_pattern = r'\b(\d)\s*star\b'
        star_matches = re.findall(star_pattern, text)
        if star_matches:
            entities['star_rating'] = star_matches
        
        # Common city names (you can expand this)
        cities = ['paris', 'london', 'cairo', 'rome', 'dubai', 'new york', 
                 'barcelona', 'tokyo', 'berlin', 'madrid', 'amsterdam']
        for city in cities:
            if city in text:
                entities['city'].append(city)
        
        # Common country names
        countries = ['france', 'egypt', 'uk', 'italy', 'spain', 'germany',
                    'usa', 'japan', 'uae']
        for country in countries:
            if country in text:
                entities['country'].append(country)
        
        return dict(entities)
    
    def match_keywords(self, text: str, intent: str) -> int:
        """
        Count keyword matches for a given intent.
        
        Args:
            text: Preprocessed text
            intent: Intent name
            
        Returns:
            Number of keyword matches
        """
        keywords = self.intent_keywords[intent]['keywords']
        matches = 0
        
        for keyword in keywords:
            # Check for whole phrase match
            if keyword in text:
                matches += 1
        
        return matches
    
    def match_patterns(self, text: str, intent: str) -> bool:
        """
        Check if text matches any regex patterns for the intent.
        
        Args:
            text: Preprocessed text
            intent: Intent name
            
        Returns:
            True if any pattern matches
        """
        patterns = self.intent_keywords[intent].get('patterns', [])
        
        for pattern in patterns:
            if re.search(pattern, text, re.IGNORECASE):
                return True
        
        return False
    
    def calculate_confidence(self, keyword_matches: int, pattern_match: bool,
                           entity_boost: float) -> float:
        """
        Calculate confidence score for intent classification.
        
        Args:
            keyword_matches: Number of keyword matches
            pattern_match: Whether pattern matched
            entity_boost: Boost from entity hints
            
        Returns:
            Confidence score (0.0 to 1.0)
        """
        confidence = 0.0
        
        # Keyword contribution (up to 0.6)
        confidence += min(keyword_matches * 0.2, 0.6)
        
        # Pattern contribution (0.3)
        if pattern_match:
            confidence += 0.3
        
        # Entity boost (up to 0.1)
        confidence += entity_boost
        
        # Cap at 1.0
        return min(confidence, 1.0)
    
    def classify(self, query: str) -> Tuple[str, float, Dict]:
        """
        Classify user query into an intent.
        
        Args:
            query: Raw user query
            
        Returns:
            Tuple of (intent, confidence, metadata)
        """
        # Preprocess
        processed_text = self.preprocess_text(query)
        
        # Extract entity hints
        entities = self.extract_entities_hints(processed_text)
        
        # Score each intent
        intent_scores = {}
        
        for intent in self.intent_priority:
            keyword_matches = self.match_keywords(processed_text, intent)
            pattern_match = self.match_patterns(processed_text, intent)
            
            # Entity boost logic
            entity_boost = 0.0
            if intent == 'DEMOGRAPHIC_RECOMMENDATION' and (entities.get('age') or 
                                                           entities.get('gender') or 
                                                           entities.get('traveler_type')):
                entity_boost = 0.1
            elif intent == 'HOTEL_SEARCH' and (entities.get('city') or 
                                               entities.get('country') or 
                                               entities.get('star_rating')):
                entity_boost = 0.05
            elif intent == 'LOCATION_QUERY' and (entities.get('city') or 
                                                 entities.get('country')):
                entity_boost = 0.05
            
            confidence = self.calculate_confidence(keyword_matches, pattern_match, 
                                                   entity_boost)
            intent_scores[intent] = confidence
        
        # Get best intent
        best_intent = max(intent_scores, key=intent_scores.get)
        best_confidence = intent_scores[best_intent]
        
        # If confidence too low, print
        if best_confidence < 0.3:
            print("low confidence")
        
        metadata = {
            'processed_query': processed_text,
            'entities': entities,
            'all_scores': intent_scores
        }
        
        return best_intent, best_confidence, metadata

In [3]:
# Example usage and testing
if __name__ == "__main__":
    classifier = HotelIntentClassifier()
    
    # Test queries
    test_queries = [
        "Find hotels in Paris",
        "Recommend a hotel for a 28-year-old female solo traveler",
        "Do I need a visa to travel from Egypt to France?",
        "What do people say about the Hilton in Cairo?",
        "Compare hotels in Rome vs Florence",
        "What's the average rating of 5-star hotels?",
        "Show me hotels with good cleanliness ratings in Dubai",
        "Which hotels do young male travelers prefer?",
        "Tell me about the Marriott Downtown",
        "Hello, can you help me?",
        "What cities have the most hotels?",
        "I'm a 45 year old business traveler, suggest hotels in London"
    ]
    
    print("=" * 80)
    print("HOTEL INTENT CLASSIFIER - TEST RESULTS")
    print("=" * 80)
    
    for query in test_queries:
        intent, confidence, metadata = classifier.classify(query)
        
        print(f"\nQuery: {query}")
        print(f"Intent: {intent}")
        print(f"Confidence: {confidence:.2f}")
        print(f"Entities: {metadata['entities']}")
        print("-" * 80)

HOTEL INTENT CLASSIFIER - TEST RESULTS

Query: Find hotels in Paris
Intent: HOTEL_SEARCH
Confidence: 0.75
Entities: {'city': ['paris']}
--------------------------------------------------------------------------------

Query: Recommend a hotel for a 28-year-old female solo traveler
Intent: DEMOGRAPHIC_RECOMMENDATION
Confidence: 0.80
Entities: {'gender': ['female'], 'traveler_type': ['solo']}
--------------------------------------------------------------------------------

Query: Do I need a visa to travel from Egypt to France?
Intent: VISA_INQUIRY
Confidence: 0.50
Entities: {'country': ['france', 'egypt']}
--------------------------------------------------------------------------------

Query: What do people say about the Hilton in Cairo?
Intent: REVIEW_QUERY
Confidence: 0.50
Entities: {'city': ['cairo']}
--------------------------------------------------------------------------------

Query: Compare hotels in Rome vs Florence
Intent: COMPARISON
Confidence: 0.70
Entities: {'city': ['rom